# Exploration 11: Build an Image Compressor

**Name:** Your name here

A photo on your phone is a grid of pixels, and a grid of numbers is a **matrix**. In this exploration you'll build an image compressor from scratch with the **singular value decomposition (SVD)**. It splits any matrix into simple **layers**, sorted from most to least important. Keep only the first few layers and you have a smaller, blurrier copy of the image.

You'll test your compressor on three images: a plain one, pure noise, and a photo of your own. Then you'll find out how it compares with the JPEG files your phone actually uses (spoiler: not great), and why.

We'll use [panchi](https://github.com/Gustavo-Galvao-e-Silva/panchi).

**Every problem has three steps:**

- **Predict:** before running any code, write down what you think will happen. Predictions are graded on being *written first*, not on being right.
- **Run:** run the code. You'll write a few short functions by filling in the blanks; the rest is written for you.
- **Explain:** compare what happened with your prediction and explain it in a sentence or two.

**How to complete this exploration**

1. Open the notebook in Google Colab and choose **File → Save a copy in Drive**, then work in that copy. (Skip this if the notebook was handed to you as your own copy, e.g. through Google Classroom.)
2. **Run the setup cell below once before beginning.** Then run the rest of the cells **from top to bottom**. Click a cell and press **Shift + Enter** to run it.
3. Wherever you see `___`, replace it with your answer and run the cell again.
4. Answer the written questions by double-clicking the *Your answer* cells and typing. Write math the way you'd type it in code, like `[1, 2, 3]` for a vector or `rank(A) = 2`. LaTeX math (`$...$`) can break the PDF export.
5. When you're done, run the **Export to PDF** cell at the very end and submit the PDF.

### Python in 60 seconds

- `x = 5` **stores** the value `5` under the name `x`. From then on, `x` means `5`.
- `print(x)` shows a value below the cell.
- Lines starting with `#` are **comments**. Python ignores them; they're notes for you.
- Python counts from **0**, not 1. The first row of a matrix is row `0`, the second is row `1`, and so on.
- `A[i, j]` is the entry of matrix `A` in row `i`, column `j`. `u[i]` is entry `i` of vector `u`.
- `x ** 2` is $x^2$, and `math.sqrt(x)` is $\sqrt{x}$.
- `def name(inputs):` makes a **function**. The indented lines below it are the function's steps, and `return` hands back its answer. **Indentation matters**: keep the spaces at the start of each line.
- `for i in range(4):` repeats the indented lines below it with `i = 0`, then `1`, `2`, `3`.
- `row.append(x)` adds `x` to the end of the list `row`. `things[:k]` is the first `k` items of a list.
- If a cell says *"This cell still has a ___ blank"*, there's still something to fill in!
- Errors are normal and nothing breaks: read the last line of the message, fix the cell, and run it again.

## ⚠️ RUN THIS SETUP CELL ONCE BEFORE BEGINNING THE EXPLORATION ⚠️

Click the ▶ button on the cell below. It takes about 30 seconds:

- A window will pop up asking for permission to access your **Google Drive**. Allow it, so the export cell at the end can find your saved notebook.
- Only run it **once**. Run it again only if Colab disconnects or you come back in a new session.

In [ ]:
# @title ⚠️ RUN THIS SETUP CELL ONCE BEFORE BEGINNING THE EXPLORATION ⚠️
# @markdown Only change this if you **renamed** your notebook:
NOTEBOOK_NAME = "exploration_svd_compression"  # @param {type:"string"}
NOTEBOOK_NAME = NOTEBOOK_NAME.strip().removesuffix(".ipynb")

import glob
import io
import json
import os
import tokenize
import uuid

# 1. Connect your Google Drive (a permission window will pop up)
from google.colab import drive

drive.mount("/content/drive")
MY_DRIVE = "/content/drive/MyDrive"

# 2. Install panchi
%pip install -q "panchi>=3.3"


# 3. Find your saved copy of this notebook in Drive, even if it's called "Copy of ..."
#    or has your name in front (the most recently saved one wins)
def find_notebook():
    found = []
    for folder in ["Colab Notebooks", "Classroom"]:
        found += glob.glob(f"{MY_DRIVE}/{folder}/**/*{NOTEBOOK_NAME}*.ipynb", recursive=True)
    return max(found, key=os.path.getmtime) if found else None


# 4. Friendly reminder instead of a confusing error when a cell still has a ___ blank
#    (IPython normally uses ___ as a name for an old output, so blanks wouldn't fail)
def _check_blanks(lines):
    try:
        tokens = tokenize.generate_tokens(io.StringIO("".join(lines)).readline)
        has_blank = any(t.type == tokenize.NAME and t.string == "___" for t in tokens)
    except Exception:
        return lines
    if has_blank:
        return ['print("This cell still has a ___ blank. '
                'Replace every ___ with your answer, then run it again.")\n']
    return lines


transformers = get_ipython().input_transformers_cleanup
transformers[:] = [t for t in transformers if getattr(t, "__name__", "") != "_check_blanks"]
transformers.append(_check_blanks)


# 5. Images and tools for the exploration
import base64
import math
import random
import struct
import subprocess
import sys

import matplotlib.cbook as cbook
import matplotlib.pyplot as plt
import panchi as pan
from google.colab import files
from IPython.display import HTML, display
from PIL import Image, ImageOps

MAX_SIDE = 128  # photos are shrunk to at most this many pixels per side, so the SVD stays quick
_PALETTE = {"bg": "#45423F", "peach": "#FFB592", "brown": "#805B49", "cream": "#F2EDE9",
            "gray": "#8F8984", "text": "#DDD6CE", "track": "#5A5652"}


def load_plain():
    """A plain 128 x 128 image: a dark background with two bars and a box."""
    size = 128
    rows = []
    for i in range(size):
        row = []
        for j in range(size):
            value = 40
            if 51 <= i < 70:
                value += 120  # the horizontal bar
            if 32 <= j < 47:
                value += 80  # the vertical bar
            if 90 <= i < 115 and 79 <= j < 109:
                value += 150  # the box
            row.append(value)
        rows.append(row)
    return pan.Matrix(rows)


def load_noise():
    """A 128 x 128 image where every pixel is a random brightness."""
    draw = random.Random(1554)
    return pan.Matrix([[draw.uniform(0, 255) for _ in range(128)] for _ in range(128)])


def _pixels(image):
    """Every pixel of a PIL image, row by row."""
    getter = getattr(image, "get_flattened_data", None) or image.getdata
    return list(getter())


def _flatten(image):
    """Put see-through images on a white background."""
    if image.mode in ("RGBA", "LA", "P", "PA"):
        image = image.convert("RGBA")
        background = Image.new("RGBA", image.size, (255, 255, 255, 255))
        image = Image.alpha_composite(background, image)
    return image


def _shrink(image):
    width, height = image.size
    if max(width, height) > MAX_SIDE:
        image = image.copy()
        image.thumbnail((MAX_SIDE, MAX_SIDE), Image.LANCZOS)
        print(f"Your photo was {width} pixels wide and {height} tall. It was shrunk to "
              f"{image.size[0]} wide and {image.size[1]} tall so the SVD stays quick.")
    else:
        print(f"Your photo is {width} pixels wide and {height} tall.")
    return image


def _open_upload():
    """Ask for a file and open it as an image, or return None with a friendly message."""
    print("Click 'Choose Files' below and pick a photo (JPG, PNG, HEIC, ...).")
    uploaded = files.upload()
    if not uploaded:
        print("No photo was uploaded. Run the cell again to try again.")
        return None
    name, data = next(iter(uploaded.items()))
    try:
        image = Image.open(io.BytesIO(data))
        image.load()
    except Exception:
        image = None
        if name.lower().endswith((".heic", ".heif")):
            try:
                try:
                    import pillow_heif
                except ImportError:
                    print("Installing HEIC support...")
                    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pillow-heif"], check=True)
                    import pillow_heif
                pillow_heif.register_heif_opener()
                image = Image.open(io.BytesIO(data))
                image.load()
            except Exception:
                image = None
        if image is None:
            print(f"'{name}' couldn't be opened as a photo. Try a JPG or PNG file instead.")
            return None
    return _flatten(ImageOps.exif_transpose(image))


def _gray_matrix(image):
    image = _shrink(image).convert("L")
    width, height = image.size
    pixels = _pixels(image)
    return pan.Matrix([[float(pixels[i * width + j]) for j in range(width)] for i in range(height)])


def _color_grid(image):
    image = _shrink(image).convert("RGB")
    width, height = image.size
    pixels = _pixels(image)
    return [[tuple(pixels[i * width + j]) for j in range(width)] for i in range(height)]


def _sample():
    return Image.open(cbook.get_sample_data("grace_hopper.jpg"))


def upload_photo():
    """Upload a photo of your own. It comes back as a gray matrix, 0 (black) to 255 (white)."""
    image = _open_upload()
    return None if image is None else _gray_matrix(image)


def sample_photo():
    """A photo to use if uploading doesn't work: computer scientist Grace Hopper."""
    return _gray_matrix(_sample())


def upload_color_photo():
    """Upload a photo of your own, in color."""
    image = _open_upload()
    return None if image is None else _color_grid(image)


def sample_color_photo():
    """A color photo to use if uploading doesn't work."""
    return _color_grid(_sample())


def _is_color(image):
    return isinstance(image, list) and image and isinstance(image[0], (list, tuple)) \
        and image[0] and isinstance(image[0][0], (list, tuple))


def _clip(x):
    return min(255.0, max(0.0, float(x)))


def _to_pil(image):
    if isinstance(image, pan.Matrix):
        rows = image.to_list()
        out = Image.new("L", (len(rows[0]), len(rows)))
        out.putdata([round(_clip(x)) for row in rows for x in row])
        return out
    if _is_color(image):
        out = Image.new("RGB", (len(image[0]), len(image)))
        out.putdata([tuple(round(_clip(c)) for c in pixel[:3]) for row in image for pixel in row])
        return out
    raise TypeError("show and save_image take a pan.Matrix (gray) or a color photo "
                    "(rows of (r, g, b) pixels).")


def _from_pil(image, like):
    width, height = image.size
    pixels = _pixels(image)
    if isinstance(like, pan.Matrix):
        return pan.Matrix([[float(pixels[i * width + j]) for j in range(width)] for i in range(height)])
    return [[tuple(pixels[i * width + j]) for j in range(width)] for i in range(height)]


def show(*images, titles=None, stretch=False):
    """Draw images side by side. stretch=True stretches the colors to fit (for single layers)."""
    count = len(images)
    if count == 0:
        return
    fig, axes = plt.subplots(1, count, figsize=(3.4 * count, 3.6), facecolor=_PALETTE["bg"], squeeze=False)
    for k, (ax, image) in enumerate(zip(axes[0], images)):
        title = titles[k] if titles and k < len(titles) else ""
        if isinstance(image, pan.Matrix):
            rows = [[float(x) for x in row] for row in image.to_list()]
            if stretch:
                low = min(min(row) for row in rows)
                high = max(max(row) for row in rows)
                if high - low < 1e-6:
                    rows = [[0.0] * len(rows[0]) for _ in rows]
                    low, high = -1.0, 1.0
                    title = (title + " (all zeros)").strip()
            else:
                low, high = 0, 255
            ax.imshow(rows, cmap="gray", vmin=low, vmax=high, interpolation="nearest")
        elif _is_color(image):
            ax.imshow([[[_clip(c) / 255 for c in pixel[:3]] for pixel in row] for row in image],
                      interpolation="nearest")
        else:
            raise TypeError("show takes pan.Matrix images (gray) or color photos (rows of (r, g, b) pixels).")
        ax.set_title(title, color=_PALETTE["cream"], fontsize=10)
        ax.axis("off")
    plt.tight_layout()
    plt.show()


def _style(ax):
    ax.set_facecolor(_PALETTE["bg"])
    ax.tick_params(colors=_PALETTE["text"])
    for spine in ax.spines.values():
        spine.set_color(_PALETTE["gray"])
    ax.xaxis.label.set_color(_PALETTE["text"])
    ax.yaxis.label.set_color(_PALETTE["text"])


def plot_singular_values(parts, log=False):
    """Bar chart of the sigmas: how big each layer of the image is."""
    sigma = list(parts.sigma)
    fig, ax = plt.subplots(figsize=(7, 3.2), facecolor=_PALETTE["bg"])
    _style(ax)
    if log:
        floor = max(sigma[0] * 1e-6, 1e-12)
        ax.bar(range(len(sigma)), [max(s, floor) for s in sigma], color=_PALETTE["peach"], width=0.8)
        ax.set_yscale("log")
        ax.set_ylabel("sigma (log scale)")
    else:
        ax.bar(range(len(sigma)), sigma, color=_PALETTE["peach"], width=0.8)
        ax.set_ylabel("sigma (size of the layer)")
    ax.set_xlabel("layer number")
    ax.set_xlim(-1, len(sigma))
    plt.tight_layout()
    plt.show()


def compare_singular_values(parts_list, labels):
    """The sigmas of several images on one log-scale plot, each divided by its biggest sigma."""
    fig, ax = plt.subplots(figsize=(7, 3.4), facecolor=_PALETTE["bg"])
    _style(ax)
    colors = [_PALETTE["peach"], _PALETTE["cream"], _PALETTE["gray"], _PALETTE["brown"]]
    for k, (parts, label) in enumerate(zip(parts_list, labels)):
        top = parts.sigma[0]
        ax.plot(range(len(parts.sigma)), [max(s / top, 1e-6) for s in parts.sigma],
                color=colors[k % len(colors)], lw=2, label=label)
    ax.set_yscale("log")
    ax.set_ylim(1e-6, 1.5)
    ax.set_xlabel("layer number")
    ax.set_ylabel("sigma / biggest sigma")
    legend = ax.legend(facecolor=_PALETTE["bg"], edgecolor=_PALETTE["gray"])
    for text in legend.get_texts():
        text.set_color(_PALETTE["cream"])
    plt.tight_layout()
    plt.show()


def save_image(image, filename="my_compressed_photo.png"):
    """Save an image as a PNG in My Drive and download it."""
    if not filename.lower().endswith(".png"):
        filename += ".png"
    path = f"{MY_DRIVE}/{filename}" if os.path.isdir(MY_DRIVE) else filename
    _to_pil(image).save(path)
    print("Saved", path)
    try:
        files.download(path)
    except Exception:
        pass


def _encoded_size(image, fmt, **options):
    buffer = io.BytesIO()
    _to_pil(image).save(buffer, fmt, **options)
    return len(buffer.getvalue())


def png_bytes(image):
    """How many bytes the image takes as a PNG file."""
    return _encoded_size(image, "PNG", optimize=True)


def jpeg_bytes(image, quality=75):
    """How many bytes the image takes as a JPEG file (quality 75 is a common setting)."""
    return _encoded_size(image, "JPEG", quality=quality)


def as_jpeg(image, quality=75):
    """The image after saving it as a JPEG and opening it again."""
    buffer = io.BytesIO()
    _to_pil(image).save(buffer, "JPEG", quality=quality)
    buffer.seek(0)
    return _from_pil(Image.open(buffer), image)


# 6. Checkers: they compare your functions with the right answers on small examples
def _close(a, b, tol=1e-6):
    return abs(float(a) - float(b)) <= tol * max(1.0, abs(float(b)))


def _same_matrix(M, N, tol=1e-6):
    return isinstance(M, pan.Matrix) and M.shape == N.shape and all(
        _close(M[i, j], N[i, j], tol) for i in range(N.rows) for j in range(N.cols))


def check_outer(outer):
    u, v = pan.Vector([1, 2]), pan.Vector([3, 4, 5])
    right = pan.Matrix([[3, 4, 5], [6, 8, 10]])
    try:
        mine = outer(u, v)
    except Exception as error:
        print("outer crashed:", repr(error))
        return
    if not isinstance(mine, pan.Matrix):
        print("outer should return a pan.Matrix. Did you forget the return line?")
    elif mine.shape != right.shape:
        print(f"outer([1, 2], [3, 4, 5]) should be 2 x 3, but yours is {mine.shape[0]} x {mine.shape[1]}.")
    elif not _same_matrix(mine, right):
        print("Not quite. outer([1, 2], [3, 4, 5]) should be:")
        print(right)
        print("Yours is:")
        print(mine)
    else:
        print("outer works!")


def check_compress(compress):
    draw = random.Random(7)
    A = pan.Matrix([[draw.randint(0, 9) for _ in range(4)] for _ in range(5)])
    parts = pan.svd(A)
    for k in range(5):
        try:
            mine = compress(parts, k)
        except Exception as error:
            print(f"compress crashed with k = {k}:", repr(error))
            return
        if not isinstance(mine, pan.Matrix):
            print("compress should return a pan.Matrix. Did you forget the return line?")
            return
        if not _same_matrix(mine, parts.approximation(k)):
            print(f"Not quite: compress(parts, {k}) isn't the sum of the first {k} layers.")
            if k == 0:
                print("With k = 0 there are no layers, so the picture should be all zeros.")
            return
    print("compress works!")


def check_storage(storage_cost):
    cases = [(128, 128, 1), (128, 128, 10), (100, 60, 7), (5, 4, 0)]
    for m, n, k in cases:
        try:
            mine = storage_cost(m, n, k)
        except Exception as error:
            print("storage_cost crashed:", repr(error))
            return
        right = k * (m + n + 1)
        if mine != right:
            print(f"Not quite: for a {m} x {n} image with k = {k}, yours says {mine}.")
            if mine == k * (m + n):
                print("Close! Each layer also needs its sigma.")
            return
    print("storage_cost works!")


def check_error(error):
    A = pan.Matrix([[3, 0], [0, 4]])
    cases = [(A, A, 0.0), (A, pan.Matrix([[3, 0], [0, 0]]), 0.8), (A, pan.Matrix([[0, 0], [0, 0]]), 1.0)]
    for X, Y, right in cases:
        try:
            mine = error(X, Y)
        except Exception as problem:
            print("error crashed:", repr(problem))
            return
        if not _close(mine, right):
            print("Not quite. For A = [[3, 0], [0, 4]] and B = [[3, 0], [0, 0]], the error should be 0.8:")
            print("the gap is sqrt(4^2) = 4 and the size of A is sqrt(3^2 + 4^2) = 5.")
            return
    print("error works!")


# 7. The layer slider: rebuilds the image from its first k layers right in your browser
def _pack(values):
    return base64.b64encode(struct.pack(f"<{len(values)}f", *values)).decode()


_LAYERS_HTML = r"""
<style>
  #BOX { background: #45423F; color: #F2EDE9; color-scheme: dark; font: 14px/1.4 system-ui, sans-serif; width: 700px; max-width: 100%; padding: 14px 16px; box-sizing: border-box; border-radius: 12px; }
  #BOX .panels { display: flex; gap: 16px; justify-content: center; flex-wrap: wrap; }
  #BOX figure { margin: 0; text-align: center; }
  #BOX canvas { image-rendering: pixelated; border-radius: 4px; display: block; }
  #BOX figcaption { color: #DDD6CE; margin-top: 4px; font-size: 13px; }
  #BOX .readout { display: flex; gap: 22px; font-variant-numeric: tabular-nums; margin: 10px 0 4px; }
  #BOX .readout b { color: #FFB592; font-weight: 600; }
  #BOX label { display: flex; gap: 10px; align-items: center; margin: 8px 0; }
  #BOX label span { width: 44px; color: #FFB592; font-weight: 600; }
  #BOX input { flex: 1; accent-color: #FFB592; }
  #BOX output { width: 50px; text-align: right; font-variant-numeric: tabular-nums; }
  #BOX button { background: #5A5652; color: #F2EDE9; border: 0; border-radius: 6px; padding: 3px 10px; font: inherit; cursor: pointer; }
  #BOX button:hover { background: #8F8984; }
</style>
<div id="BOX"></div>
<script>
(() => {
  const d = DATA;
  const box = document.getElementById(d.box);
  const m = d.m, n = d.n, r = d.sigma.length, S = d.sigma;
  const unpack = s => { const b = atob(s), a = new Uint8Array(b.length); for (let i = 0; i < b.length; i++) a[i] = b.charCodeAt(i); return new Float32Array(a.buffer); };
  const U = unpack(d.u), V = unpack(d.v), A = unpack(d.a);
  const scale = 310 / Math.max(m, n), w = Math.round(n * scale), h = Math.round(m * scale);
  box.innerHTML = `<div class="panels">
      <figure><canvas class="orig" width="${n}" height="${m}" style="width:${w}px;height:${h}px"></canvas><figcaption>Original</figcaption></figure>
      <figure><canvas class="approx" width="${n}" height="${m}" style="width:${w}px;height:${h}px"></canvas><figcaption class="cap"></figcaption></figure>
    </div>
    <div class="readout"><div>Layers kept <b class="k"></b> of ${r}</div><div>sigma of the last layer added <b class="sig"></b></div></div>
    <label><span>k</span><button class="play">Play</button><button class="minus">-1</button><input class="step" type="range" min="0" max="${r}" step="1" value="0"><button class="plus">+1</button><output class="kout"></output></label>`;
  const $ = q => box.querySelector(q);
  const paint = (canvas, values) => {
    const ctx = canvas.getContext("2d"), img = ctx.createImageData(n, m);
    for (let p = 0; p < m * n; p++) {
      const v = Math.max(0, Math.min(255, Math.round(values[p])));
      img.data[4 * p] = img.data[4 * p + 1] = img.data[4 * p + 2] = v; img.data[4 * p + 3] = 255;
    }
    ctx.putImageData(img, 0, 0);
  };
  paint($(".orig"), A);
  const acc = new Float64Array(m * n);
  let have = 0;
  const layer = (l, sign) => {
    const s = sign * S[l];
    for (let i = 0; i < m; i++) {
      const ui = s * U[i * r + l];
      if (!ui) continue;
      for (let j = 0; j < n; j++) acc[i * n + j] += ui * V[j * r + l];
    }
  };
  const setK = k => {
    if (k === 0) { acc.fill(0); have = 0; return; }
    while (have < k) layer(have++, 1);
    while (have > k) layer(--have, -1);
  };
  const slider = $(".step");
  const show = () => {
    const k = Number(slider.value);
    setK(k);
    paint($(".approx"), acc);
    $(".k").textContent = k; $(".kout").textContent = k;
    $(".cap").textContent = `First ${k} layer${k === 1 ? "" : "s"}`;
    $(".sig").textContent = k ? S[k - 1].toFixed(1) : "-";
  };
  slider.addEventListener("input", show);
  $(".minus").addEventListener("click", () => { slider.value = Number(slider.value) - 1; show(); });
  $(".plus").addEventListener("click", () => { slider.value = Number(slider.value) + 1; show(); });
  let timer = null;
  const stop = () => { clearInterval(timer); timer = null; $(".play").textContent = "Play"; };
  $(".play").addEventListener("click", () => {
    if (timer) return stop();
    if (Number(slider.value) >= r) slider.value = 0;
    $(".play").textContent = "Pause";
    timer = setInterval(() => {
      if (Number(slider.value) >= r) return stop();
      slider.value = Number(slider.value) + 1;
      show();
    }, 150);
  });
  show();
})();
</script>
"""


def explore_layers(parts):
    """A slider that rebuilds the image from its first k layers."""
    m, n = parts.original.shape
    u_rows, v_rows = parts.u.to_list(), parts.v.to_list()
    data = {
        "box": f"layers{uuid.uuid4().hex}", "m": m, "n": n,
        "sigma": [float(s) for s in parts.sigma],
        "u": _pack([float(x) for row in u_rows for x in row]),
        "v": _pack([float(x) for row in v_rows for x in row]),
        "a": _pack([float(x) for row in parts.original.to_list() for x in row]),
    }
    display(HTML(_LAYERS_HTML.replace("BOX", data["box"]).replace("DATA", json.dumps(data))))


if find_notebook():
    print("Setup complete! You can start the exploration now.")
else:
    print("Setup complete, but your saved copy of this notebook wasn't found in Drive.")
    print("Choose File > Save a copy in Drive, then work in (and run setup in) that copy.")

In [ ]:
import math

import panchi as pan

## Problem 1. An image is a matrix

A grayscale image is a matrix: one entry per pixel, from **0 (black)** to **255 (white)**. Here's our plain image, 128 pixels by 128 pixels. That's $128 \times 128 = 16384$ numbers.

In [ ]:
A = load_plain()
print("Size:", A.shape)
show(A)

Pick a few pixels and look at their numbers. Try one in the dark background, one in a bar, one where the two bars cross, and one in the box. Row `0` is at the top and column `0` is at the left.

In [ ]:
print(A[___, ___])
print(A[___, ___])
print(A[___, ___])
print(A[___, ___])

Each **row** of the matrix is one horizontal line of pixels. Going down the image, the rows change only at a few places.

**Predict:** how many **different kinds** of rows does this image have? Can every row be written as a combination of just a few of them? Based on that, what do you think the **rank** of $A$ is?

*Type math like code, e.g. `[1, 2, 3]` or `rank(A) = 2`, not LaTeX.*

*(write your prediction here **before** running the next cell)*

## Problem 2. Layers: the simplest images

The simplest image you can build from two vectors $\vec u$ (one entry per row) and $\vec v$ (one entry per column) is their **outer product** $\vec u \vec v^T$. Its entry in row $i$, column $j$ is

$$(\vec u \vec v^T)_{ij} = u_i \, v_j.$$

We'll call an image like this a **layer**. Your first job: write the function that builds one. Fill in the entry for row `i`, column `j`.

In [ ]:
def outer(u, v):
    """The layer u v^T: its entry in row i, column j is u[i] times v[j]."""
    rows = []
    for i in range(len(u)):
        row = []
        for j in range(len(v)):
            row.append(___)
        rows.append(row)
    return pan.Matrix(rows)


check_outer(outer)

Here's a small layer. $\vec u$ says which rows light up and $\vec v$ says which columns light up. We multiply by 200 to make it bright.

In [ ]:
u = pan.Vector([0, 1, 1, 1, 1, 0])
v = pan.Vector([0, 0, 1, 1, 1, 1, 1, 0])

layer = outer(u, v)
print(layer)
show(200 * layer)

Now draw a **plus sign** on a 7 by 7 grid: a horizontal bar across the middle row, and a vertical bar down the middle column. Each bar is one layer. Type 7 numbers in each list.

In [ ]:
across = outer(pan.Vector([___]), pan.Vector([___]))  # the horizontal bar
down = outer(pan.Vector([___]), pan.Vector([___]))    # the vertical bar

show(200 * across, 200 * down, 200 * (across + down), titles=["across", "down", "across + down"])

**Explain:** look at the printed layer above. Every row of $\vec u \vec v^T$ is a multiple of which vector? So what is the rank of a layer? Why can **no single layer** draw the plus sign, however you choose $\vec u$ and $\vec v$? How many layers would you need for the plain image from Problem 1?

*Type math like code, e.g. `[1, 2, 3]` or `rank(A) = 2`, not LaTeX.*

*(write your answer here)*

## Problem 3. The SVD peels an image into layers

Any matrix, including any image, can be written as a sum of layers. The **SVD** finds the best way to do it:

$$A = \sigma_0 \, \vec u_0 \vec v_0^T + \sigma_1 \, \vec u_1 \vec v_1^T + \sigma_2 \, \vec u_2 \vec v_2^T + \cdots$$

- The vectors $\vec u_0, \vec u_1, \dots$ are **orthonormal**: unit length and perpendicular to each other. So are $\vec v_0, \vec v_1, \dots$
- Since the vectors all have length 1, the number $\sigma_i$ (sigma, a **singular value**) says how **big** layer $i$ is.
- The SVD sorts the layers from biggest to smallest: $\sigma_0 \ge \sigma_1 \ge \sigma_2 \ge \cdots \ge 0$.

`pan.svd(A)` computes all of this. In what follows, `parts.sigma` is the list of sigmas, and `parts.pairs[i]` is layer `i` as `(sigma, u, v)`.

`parts` also has shortcuts that do the rest of this exploration for you. **Don't use them**: the point is to build the compressor yourself, from `parts.pairs` and `parts.sigma` only.

In [ ]:
parts = pan.svd(A)

**Predict:** the bar chart below shows the size $\sigma$ of every layer of the plain image. How many bars will be bigger than zero? Use your answer to Problem 1.

*Type math like code, e.g. `[1, 2, 3]` or `rank(A) = 2`, not LaTeX.*

*(write your prediction here **before** running the next cell)*

In [ ]:
plot_singular_values(parts)
print("The first 5 sigmas:", [round(s, 1) for s in parts.sigma[:5]])

Now look at the layers one at a time. A single layer can have negative numbers, so `stretch=True` stretches its colors to fit. Try `0`, `1`, `2` and `3`.

In [ ]:
sigma, u, v = parts.pairs[___]  # try 0, 1, 2, 3

print("sigma =", round(sigma, 2))
show(sigma * outer(u, v), stretch=True)

**Explain:** how many bars were bigger than zero, and how does that match your rank from Problem 1? Describe what layers 0, 1 and 2 look like. What happened at layer 3, and why?

*Type math like code, e.g. `[1, 2, 3]` or `rank(A) = 2`, not LaTeX.*

*(write your answer here)*

## Problem 4. Build the compressor

Here's the whole idea of SVD compression: **keep only the first $k$ layers.**

$$A_k = \sigma_0 \, \vec u_0 \vec v_0^T + \cdots + \sigma_{k-1} \, \vec u_{k-1} \vec v_{k-1}^T$$

Since the layers are sorted, you keep the biggest ones. In fact, a famous theorem (**Eckart–Young**) says that no picture made of $k$ layers is closer to $A$ than $A_k$.

### (a) Compress

`compress` starts from an all-black image and adds the first `k` layers, one at a time.

In [ ]:
def compress(parts, k):
    """Add up the first k layers of the image."""
    m, n = parts.original.shape
    image = pan.zero_matrix(m, n)
    for sigma, u, v in parts.pairs[:k]:
        image = image + ___
    return image


check_compress(compress)
show(A, compress(parts, 1), compress(parts, 2), compress(parts, 3), titles=["original", "k = 1", "k = 2", "k = 3"])

### (b) Storage

To store $k$ layers of an $m \times n$ image, you keep, for each layer, the numbers in $\vec u$, the numbers in $\vec v$ and the number $\sigma$. The original image stores $m \times n$ numbers.

In [ ]:
def storage_cost(m, n, k):
    """How many numbers it takes to store k layers of an m x n image."""
    return ___


check_storage(storage_cost)
m, n = A.shape
print("Original:", m * n, "numbers")
print("3 layers:", storage_cost(m, n, 3), "numbers")

### (c) Error

How far is the compressed image $B$ from the original $A$? Add up the squared differences of all the pixels, take the square root, and divide by the size of $A$ the same way:

$$\text{error} = \frac{\sqrt{\sum_{i,j} (a_{ij} - b_{ij})^2}}{\sqrt{\sum_{i,j} a_{ij}^2}}$$

An error of `0` means a perfect copy, and `0.05` means about 5% off. Fill in what each pixel adds to `top` and to `bottom`.

In [ ]:
def error(A, B):
    """How far B is from A, as a fraction of the size of A."""
    m, n = A.shape
    top = 0     # adds up (a_ij - b_ij)^2
    bottom = 0  # adds up a_ij^2
    for i in range(m):
        for j in range(n):
            top = top + ___
            bottom = bottom + ___
    return ___


check_error(error)
for k in range(4):
    print("k =", k, " error =", round(error(A, compress(parts, k)), 6))

### (d) The sigmas are the information

Your `error` looked at all 16384 pixels. The cell below computes something that only uses the sigmas: the square root of the sum of the squared sigmas you **threw away**, divided by the same thing for **all** the sigmas.

**Predict:** will this number be close to your `error`, or different?

*Type math like code, e.g. `[1, 2, 3]` or `rank(A) = 2`, not LaTeX.*

*(write your prediction here **before** running the next cell)*

In [ ]:
for k in range(4):
    thrown_away = math.sqrt(sum(s ** 2 for s in parts.sigma[k:]))
    everything = math.sqrt(sum(s ** 2 for s in parts.sigma))
    print("k =", k, " your error =", round(error(A, compress(parts, k)), 6), " from the sigmas =", round(thrown_away / everything, 6))

**Explain:** what fraction of the original numbers does the 3-layer version store? Why is it a perfect copy? What did you notice about the two error columns, and what does that tell you about what the sigmas measure?

*Type math like code, e.g. `[1, 2, 3]` or `rank(A) = 2`, not LaTeX.*

*(write your answer here)*

## Problem 5. Noise

Now the opposite of a plain image: every pixel is a **random** brightness, with no pattern at all.

In [ ]:
N = load_noise()
show(N)
parts_noise = pan.svd(N)

**Predict:** what will the bar chart of sigmas look like for noise? How many layers will you need before it looks like the original?

*Type math like code, e.g. `[1, 2, 3]` or `rank(A) = 2`, not LaTeX.*

*(write your prediction here **before** running the next cell)*

In [ ]:
plot_singular_values(parts_noise)

Layer 0 is much bigger than the rest. Look at it.

In [ ]:
sigma, u, v = parts_noise.pairs[0]
layer0 = sigma * outer(u, v)
print("Some pixels of layer 0:", round(layer0[0, 0]), round(layer0[50, 90]), round(layer0[127, 3]))
show(layer0, stretch=True)

Drag the slider from 0 up to 128 layers, slowly. **Play** adds one layer at a time.

In [ ]:
# @title Sliders { display-mode: "form" }
explore_layers(parts_noise)

Compression only helps while it stores **fewer** numbers than the original. Find the **break-even** point: the biggest $k$ where `storage_cost(128, 128, k)` is still less than $128 \times 128$. (*Hint:* you can type math, like `16384 / 257`, in a cell.)

In [ ]:
k_even = ___  # the biggest k that still stores fewer numbers than the original

print("Storage:", storage_cost(128, 128, k_even), "numbers, vs", 128 * 128, "for the original")
B = compress(parts_noise, k_even)
print("Error:", round(error(N, B), 3))
show(N, B, titles=["noise", f"k = {k_even}"])

**Explain:** what is layer 0 of the noise, and why is it so much bigger than the others? (*Hint:* what's the average pixel?) After layer 0, how do the sigmas compare with the plain image's? At the break-even $k$, does the compressed noise look like the original? Why can no layer capture much of the noise?

*Type math like code, e.g. `[1, 2, 3]` or `rank(A) = 2`, not LaTeX.*

*(write your answer here)*

## Problem 6. Your photo

Upload a photo of your own: anything works, like a pet, a building or a friend (with their permission). It will be turned gray and shrunk to at most 128 pixels per side, so the SVD stays quick. If uploading doesn't work, use `photo = sample_photo()` instead.

In [ ]:
photo = upload_photo()
show(photo)

In [ ]:
parts_photo = pan.svd(photo)

**Predict:** will your photo's sigmas drop off like the plain image's, stay flat like the noise's, or something in between? Why?

*Type math like code, e.g. `[1, 2, 3]` or `rank(A) = 2`, not LaTeX.*

*(write your prediction here **before** running the next cell)*

This plot draws all three images' sigmas together. Each is divided by its biggest sigma, and the scale is **logarithmic**: each step up the axis is 10 times bigger.

In [ ]:
compare_singular_values([parts, parts_noise, parts_photo], ["plain", "noise", "your photo"])

Drag the slider and find the **smallest** $k$ where your photo looks fine to you (you can still tell it's compressed, but it's clearly the same picture).

In [ ]:
# @title Sliders { display-mode: "form" }
explore_layers(parts_photo)

In [ ]:
my_k = ___  # your smallest k that looks fine

m, n = photo.shape
B = compress(parts_photo, my_k)
print("Storage:", storage_cost(m, n, my_k), "numbers, vs", m * n, "for the original")
print("That's", round(100 * storage_cost(m, n, my_k) / (m * n)), "% of the original")
print("Error:", round(error(photo, B), 3))
show(photo, B, titles=["your photo", f"k = {my_k}"])
save_image(B, "my_compressed_photo.png")

**Explain:** where did your photo land between the plain image and the noise on the plot? What do the first few layers of your photo capture, and what do the last ones add? Why does a real photo compress much better than noise, but worse than the plain image?

*Type math like code, e.g. `[1, 2, 3]` or `rank(A) = 2`, not LaTeX.*

*(write your answer here)*

## Problem 7. Reality check: is this a good compressor?

So far we've counted **numbers**. But a computer stores **bytes**, and the two kinds of numbers aren't the same size:

- A pixel is a whole number from 0 to 255, which fits in **1 byte**.
- $\sigma$ and the entries of $\vec u$ and $\vec v$ are decimals, like `0.0831`. Each takes **4 bytes**.

Your phone doesn't store photos as raw pixels either. It uses formats like **PNG** (exact) and **JPEG** (a lossy format, like ours).

**Predict:** will your compressed photo be a smaller file than the raw pixels? Than a PNG? Than a JPEG?

*Type math like code, e.g. `[1, 2, 3]` or `rank(A) = 2`, not LaTeX.*

*(write your prediction here **before** running the next cell)*

In [ ]:
def storage_bytes(m, n, k):
    """How many bytes k layers take, at 4 bytes per number."""
    return ___


m, n = photo.shape
print("Raw pixels:", m * n, "bytes")
print(f"Your SVD, k = {my_k}:", storage_bytes(m, n, my_k), "bytes")
print("PNG:", png_bytes(photo), "bytes")
print("JPEG:", jpeg_bytes(photo), "bytes")
print("JPEG error:", round(error(photo, as_jpeg(photo)), 3))
show(photo, compress(parts_photo, my_k), as_jpeg(photo), titles=["your photo", f"SVD, k = {my_k}", "JPEG"])

Now find the $k$ where your SVD's error is about as small as the JPEG's. Try a few values.

In [ ]:
k_jpeg = ___  # try a few values until the error matches the JPEG's

print("SVD error:", round(error(photo, compress(parts_photo, k_jpeg)), 3), " JPEG error:", round(error(photo, as_jpeg(photo)), 3))
print("SVD:", storage_bytes(m, n, k_jpeg), "bytes   JPEG:", jpeg_bytes(photo), "bytes")

**Explain:** was your prediction right? How many times bigger is the SVD than the JPEG at the same error? Use these facts to explain why the SVD loses:

- A JPEG builds every image from the **same** fixed set of patterns (waves of different sizes), so it never stores the patterns themselves. What does the SVD have to store for every layer?
- A JPEG works on small **8 × 8 blocks** of the image. How far across the image does each SVD layer reach? What does one sharp edge cost each of them?
- Is there any image in this exploration where the SVD would beat a JPEG? Why that one?

*Type math like code, e.g. `[1, 2, 3]` or `rank(A) = 2`, not LaTeX.*

*(write your answer here)*

**The takeaway.** The SVD gives the *best* picture you can build from $k$ layers, but "best $k$ layers" isn't the same as "smallest file". Real image formats are built on fixed, local patterns and clever rounding. The SVD's real strength is **finding structure**: the few directions that matter most in a big table of numbers. That's what it's used for, in face recognition, Netflix-style recommendations, removing noise from data, and the PCA method in statistics and machine learning.

## Challenge (Optional): color

Your compressor works on a `pan.Matrix` where every pixel is **one** number. A color photo is different: every pixel is **three** numbers, how much red, green and blue it has, each from 0 to 255.

`upload_color_photo()` gives you a color photo (or use `sample_color_photo()`). `photo_c[i][j]` is the pixel in row `i`, column `j`, like `(212, 140, 97)`. `show` and `save_image` work on color photos too.

**Your task:** make a color version of your compressor. It should take a color photo and a number of layers, and give back a color photo you can `show`. This time there are no blanks; write it yourself.

In [ ]:
photo_c = upload_color_photo()
print("Rows:", len(photo_c), " Columns:", len(photo_c[0]))
print("Top-left pixel:", photo_c[0][0])
show(photo_c)

**Before you code: your plan.** Describe in words how you'll use what you've built to compress a color photo. How many numbers will your compressed photo take at $k$ layers, compared with the original?

*Type math like code, e.g. `[1, 2, 3]` or `rank(A) = 2`, not LaTeX.*

*(write your plan here **before** writing your code)*

In [ ]:
# Build your color compressor here

In [ ]:
# Try it on your photo at a few values of k, and show the results

**Your results:** which $k$ looks fine in color, and how does it compare with your gray $k$ from Problem 6? How many numbers and bytes does it take, and how does that compare with a JPEG (`jpeg_bytes(photo_c)`)? At small $k$, do you see any strange colors? Where do they come from? Is there more than one way to turn a color photo into matrices for the SVD? Which way stores fewer numbers?

*Type math like code, e.g. `[1, 2, 3]` or `rank(A) = 2`, not LaTeX.*

*(write your answer here)*

## Export to PDF

When you've finished everything above:

1. **Save** the notebook (**File → Save**, or Ctrl/Cmd + S). The PDF is made from the *saved* copy in your Drive.
2. Click the ▶ button on the cell below. The first time, it spends a few minutes installing the PDF tools before making your PDF.

Your PDF will download to your computer and also be saved in the main folder of your Google Drive (**My Drive**). The setup and export cells won't show up in the PDF.

In [ ]:
# @title Export to PDF
import shutil

from google.colab import files


def pdf_tools_ready():
    return shutil.which("xelatex") is not None and shutil.which("pandoc") is not None


# Install the PDF tools (LaTeX + pandoc) the first time this runs. It takes a few minutes.
if not pdf_tools_ready():
    print("Installing the PDF tools. This takes a few minutes, but only the first time...")
    !apt-get update -qq > /dev/null
    !DEBIAN_FRONTEND=noninteractive apt-get install -y -qq texlive-xetex texlive-fonts-recommended texlive-latex-extra texlive-plain-generic pandoc > /dev/null

NOTEBOOK = find_notebook()

if not pdf_tools_ready():
    print("Installing the PDF tools failed (see the messages above). Run this cell again.")
elif NOTEBOOK is None:
    print("Couldn't find your saved copy of this notebook in Drive.")
    print("Choose File > Save a copy in Drive, then work in that copy.")
else:
    # Copy the saved notebook out of Drive and convert it here. The PDF shows code and
    # outputs, hides the In [ ]: prompts, and leaves out the setup and export cells.
    LOCAL = f"/content/{NOTEBOOK_NAME}.ipynb"
    PDF = f"/content/{NOTEBOOK_NAME}.pdf"
    !rm -f "{PDF}"
    !cp "{NOTEBOOK}" "{LOCAL}"
    !jupyter nbconvert --to PDF "{LOCAL}" --log-level=ERROR --TemplateExporter.exclude_input_prompt=True --TemplateExporter.exclude_output_prompt=True --TagRemovePreprocessor.enabled=True --TagRemovePreprocessor.remove_cell_tags=remove_cell --TagRemovePreprocessor.remove_input_tags=remove_input

    if os.path.exists(PDF):
        !cp "{PDF}" "{MY_DRIVE}/"
        print("Done! Your PDF is in My Drive and is downloading now:", os.path.basename(PDF))
        files.download(PDF)
    else:
        print("Something went wrong making the PDF; see the messages above.")